# Nifty 50 Sector Analysis

This notebook demonstrates two key analyses for the **Historical Nifty 50 Constituent Weights** dataset:

1.  **Sector Rotation:** We'll visualize how the weight composition of major sectors has evolved over time.
2.  **Sector Concentration:** We'll analyze if the index has become more or less concentrated in its top sectors.

## Part 1: Sector Rotation Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

# Load the datasets
weights_df = pd.read_csv('weights.csv', index_col=0, parse_dates=True)
sectors_df = pd.read_csv('sectors.csv')

# Prepare the ticker-to-sector mapping dictionary
ticker_to_sector = pd.Series(sectors_df.SECTOR.values, index=sectors_df.STOCK).to_dict()

# Calculate the historical weights for each sector
sectors = sorted(list(set(ticker_to_sector.values())))
sector_weights_df = pd.DataFrame(index=weights_df.index)
for sector in sectors:
    relevant_tickers = [t for t in ticker_to_sector if ticker_to_sector[t] == sector and t in weights_df.columns]
    sector_weights_df[sector] = weights_df[relevant_tickers].sum(axis=1)

print("Sector weights calculated successfully.")

# To reduce chart clutter, we are grouping smaller sectors into an 'OTHERS' category
avg_weights = sector_weights_df.mean()
major_sectors = avg_weights[avg_weights >= 3.0].index.tolist() # Threshold of 3%
minor_sectors = avg_weights[avg_weights < 3.0].index.tolist()

clean_sector_weights_df = sector_weights_df[major_sectors].copy()
clean_sector_weights_df['OTHERS'] = sector_weights_df[minor_sectors].sum(axis=1)

print(f"Grouped {len(minor_sectors)} minor sectors into 'OTHERS'.")

# Creating the stacked area chart
fig, ax = plt.subplots(figsize=(20, 10))
clean_sector_weights_df.plot(kind='area', stacked=True, ax=ax, colormap='tab20')

ax.set_title('Historical Sector Weight Composition of Nifty 50', fontsize=20, pad=20)
ax.set_ylabel('Total Weight in Index (%)', fontsize=14)
ax.set_xlabel('Year', fontsize=14)
ax.legend(title='Sectors', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

This chart shows the main trends for dominant sectors like Banking, IT, and Energy, while neatly bundling the smaller ones.

---

## Part 2: Sector Concentration Analysis (HHI)

In [ ]:
"""The Herfindahl-Hirschman Index (HHI) is calculated by squaring the market share (or weight) of each firm
(or in our case, sector) and then summing the resulting numbers."""

hhi = (sector_weights_df**2).sum(axis=1)

# Plotting the HHI over time
fig, ax = plt.subplots(figsize=(20, 8))
hhi.plot(ax=ax, color='navy', linewidth=2.5)

ax.set_title('Nifty 50 Sector Concentration (HHI) Over Time', fontsize=20, pad=20)
ax.set_ylabel('HHI Score', fontsize=14)
ax.set_xlabel('Year', fontsize=14)
ax.grid(True, which='both', linestyle='--', linewidth=0.5)

print("HHI calculated and plotted.")
plt.show()

### Interpreting the HHI Chart

This chart can be interpreted as:
- **An upward-sloping line** indicates that the Nifty 50 is becoming more concentrated in a few dominant sectors.
- **A downward-sloping line** suggests the index is becoming more diversified across different sectors.